
# Predicción de victorias/derrotas con métricas ROLL10

En este cuaderno construiremos un flujo completo y fácilmente extensible para predecir si el equipo local ganará o perderá un partido (`W/L`). El objetivo es mantener un **pipeline didáctico**, apoyándonos en las métricas rodantes de los últimos 10 encuentros (`ROLL10_…`) y transformándolas a un **dataset a nivel de partido (home vs away)** con características relativas.

Los pasos principales incluyen: (1) recalcular las medias rodantes sin fuga temporal mediante `shift(1)`, (2) generar el dataset home/away con variables diferenciales entre ambos equipos y (3) entrenar un modelo **XGBoost** con *early stopping*. Además, seleccionaremos el umbral óptimo en validación con el criterio de **Youden**, representaremos las métricas clave y exportaremos las predicciones para su análisis posterior.



In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss, log_loss,
    confusion_matrix, roc_curve, precision_recall_curve
)
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve

try:
    from xgboost import XGBClassifier
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', '-q'])
    from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')




## Configuración principal

La celda siguiente concentra todas las opciones importantes del flujo: rutas de entrada/salida, ventana rolling, lista de métricas base y parámetros de muestreo. Modificando este bloque se pueden probar nuevas combinaciones de características, activar fuentes externas o ajustar la semilla de aleatoriedad.



In [ ]:

# === CONFIGURACIÓN PRINCIPAL ===
DATA_PATH = "/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/teamgamelogs_by_game.parquet"
OUTPUT_DIR = "/Users/pablo/Documents/BigData/BasketballAnalysis/02_processing_data/02a_WL_prediction/preds"
ROLLING_WINDOW = 10
BASE_FEATURES = [
    'NET_RATING','E_NET_RATING','DEF_RATING','E_DEF_RATING',
    'PTS','FGM','FG_PCT','EFG_PCT','TS_PCT',
    'TOV','TM_TOV_PCT','OREB_PCT','DREB','POSS',
    'PACE','PCT_FGA_2PT','PCT_FGA_3PT','PCT_PTS_FT'
]
RELATIVE_PREFIXES = ("ROLL10_", "OFF_", "DEF_", "NET_", "PACE", "E_", "USG", "PTS", "REB", "AST")
TEST_SIZE = 0.20
VAL_SIZE = 0.10
RANDOM_STATE = 42

# === OPCIONAL: Parquets externos ===
# ExternalFeaturesConfig = [
#     {
#         "path": "/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/dashboards/team_dash_pt_pass__dataset_0.parquet",
#         "on": ["TEAM_ID","GAME_DATE"],
#         "select": ["AST_PCT","PCT_AST_2PM","PCT_AST_3PM"],
#         "prefix": "PASS_"
#     }
# ]
ExternalFeaturesConfig = []




## Carga y limpieza de datos

Cargamos el parquet indicado en la configuración y nos aseguramos de que las columnas fundamentales estén presentes. Convertimos `GAME_DATE` a formato `datetime`, generamos `WL_NUM` si solo contamos con la etiqueta textual y dejamos únicamente identificadores más las métricas base que alimentarán el cálculo rolling y las diferencias home/away.



In [ ]:

df = pd.read_parquet(DATA_PATH)
df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])

if 'WL_NUM' not in df.columns and 'WL' in df.columns:
    df['WL_NUM'] = df['WL'].map({'W': 1, 'L': 0})
if 'WL' not in df.columns and 'WL_NUM' in df.columns:
    df['WL'] = df['WL_NUM'].map({1: 'W', 0: 'L'})

id_columns = ['TEAM_ID', 'TEAM_ABBREVIATION', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'WL_NUM']
available_base = [c for c in BASE_FEATURES if c in df.columns]
missing_base = sorted(set(BASE_FEATURES) - set(available_base))
if missing_base:
    print(f"Aviso: se omiten columnas base no disponibles: {missing_base}")

selected_columns = [c for c in id_columns if c in df.columns] + available_base
df = df[selected_columns].copy()

print(f"Filas totales: {len(df)}")
print(f"Fechas: {df['GAME_DATE'].min().date()} → {df['GAME_DATE'].max().date()}")
print(f"Equipos únicos: {df['TEAM_ID'].nunique()}")
df.head(3)




## Añadir features externas (opcional)

Si deseamos enriquecer el dataset con variables adicionales, podemos utilizar la estructura `ExternalFeaturesConfig`. Cada elemento define la ruta del parquet, las claves de unión, las columnas que queremos incorporar y un prefijo para renombrarlas. El siguiente ejemplo permanece comentado para mantener el flujo base ligero, pero sirve como plantilla para añadir más información cuando sea necesario.



In [ ]:

# def add_external_features(df, cfg):
#     ext = pd.read_parquet(cfg["path"])
#     ext = ext[cfg["on"] + cfg["select"]]
#     new_cols = [cfg["prefix"] + c for c in cfg["select"]]
#     ext.columns = cfg["on"] + new_cols
#     return df.merge(ext, on=cfg["on"], how="left")

# for cfg in ExternalFeaturesConfig:
#     df = add_external_features(df, cfg)
# print("Datos extendidos con features externas (si las hubiera).")




## Cálculo de métricas rolling sin fuga

Ordenamos por equipo y fecha, aplicando `shift(1)` antes de la media móvil para evitar fuga temporal. De este modo cada fila solo utiliza información de partidos anteriores al actual.



In [ ]:

df = df.sort_values(['TEAM_ID', 'GAME_DATE'])
base_cols = [c for c in BASE_FEATURES if c in df.columns]

for col in base_cols:
    roll_col = f"ROLL10_{col}"
    df[roll_col] = (
        df.groupby('TEAM_ID')[col]
          .shift(1)
          .rolling(ROLLING_WINDOW)
          .mean()
          .reset_index(level=0, drop=True)
    )

generated_cols = [f"ROLL10_{c}" for c in base_cols if f"ROLL10_{c}" in df.columns]
print(f"Columnas ROLL10 creadas: {len(generated_cols)}")




## Imputación de valores nulos

Completamos los `NaN` de las métricas rolling utilizando la mediana de cada columna. Esto garantiza que la construcción posterior del dataset home/away no propague valores faltantes al calcular diferencias.



In [ ]:

roll_cols = sorted([c for c in df.columns if c.startswith('ROLL10_')])
for col in roll_cols:
    df[col] = df[col].fillna(df[col].median())

print(f"Columnas imputadas: {len(roll_cols)}")




## Construcción del dataset local vs visitante

Transformamos el registro por equipo en un dataset a nivel de partido. Identificamos al equipo local a partir de `MATCHUP`, separamos locales y visitantes y, tras cruzarlos por `GAME_ID`, generamos características diferenciales (`HOME_x - AWAY_x`). El objetivo (`y`) se define desde la perspectiva local.



In [ ]:

def build_match_level(df, numeric_feature_prefixes=RELATIVE_PREFIXES):
    d = df.copy()
    d['IS_HOME'] = d['MATCHUP'].astype(str).str.contains('vs', case=False).astype(int)

    home = d[d['IS_HOME'] == 1].copy()
    away = d[d['IS_HOME'] == 0].copy()

    home = home.add_prefix('HOME_')
    away = away.add_prefix('AWAY_')

    merged = pd.merge(
        home, away,
        left_on='HOME_GAME_ID', right_on='AWAY_GAME_ID',
        how='inner', suffixes=('', '')
    )

    merged['y'] = (merged['HOME_WL'].astype(str).str.upper().str.strip() == 'W').astype(int)

    candidates = []
    for col in merged.columns:
        if col.startswith(('HOME_', 'AWAY_')) and merged[col].dtype.kind in 'if':
            base = col.replace('HOME_', '').replace('AWAY_', '')
            if base.startswith(numeric_feature_prefixes):
                candidates.append(base)

    candidates = sorted(set(candidates))

    X_rel = pd.DataFrame(index=merged.index)
    for base in candidates:
        h = f'HOME_{base}'
        a = f'AWAY_{base}'
        if h in merged.columns and a in merged.columns:
            X_rel[f'DIFF_{base}'] = merged[h] - merged[a]

    X_rel['HOME_COURT'] = 1
    y = merged['y'].values

    aux_cols = ['HOME_TEAM_ABBREVIATION', 'AWAY_TEAM_ABBREVIATION', 'HOME_GAME_DATE']
    aux_cols = [c for c in aux_cols if c in merged.columns]
    meta = merged[aux_cols].copy()

    return X_rel, y, meta

X_all, y_all, meta_all = build_match_level(df)
print(f"Shape X: {X_all.shape}")
print(f"Tasa de victoria local: {y_all.mean():.3f}")
X_all.head()




## Entrenamiento con XGBoost y validación temporal

Dividimos el dataset en conjuntos de entrenamiento y prueba (80/20) estratificados, reservando además un 10 % del entrenamiento como validación para *early stopping*. Ajustamos un `XGBClassifier` conservador y calculamos el umbral óptimo (Youden) en la validación antes de generar predicciones finales sobre el test.



In [ ]:

X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
    X_all, y_all, meta_all,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_all
)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_train
)

xgb = XGBClassifier(
    n_estimators=1200,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    min_child_weight=2,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False,
    early_stopping_rounds=75
)

proba_val = xgb.predict_proba(X_val)[:, 1]
proba_test = xgb.predict_proba(X_test)[:, 1]

fpr_val, tpr_val, thresholds_val = roc_curve(y_val, proba_val)
youden_scores = tpr_val - fpr_val
best_idx = int(np.argmax(youden_scores))
best_thresh = float(thresholds_val[best_idx])
if not np.isfinite(best_thresh):
    best_thresh = 0.5

y_pred = (proba_test >= best_thresh).astype(int)
cm = confusion_matrix(y_test, y_pred)

print(f"Partidos Train/Test: {len(X_train)} / {len(X_test)}")
print(f"Victorias locales en train: {y_train.mean():.3f}")
print(f"Umbral óptimo (Youden): {best_thresh:.4f}")




## Evaluación con métricas clasificatorias

Calculamos las métricas principales sobre el conjunto de test utilizando el umbral de Youden obtenido en validación: exactitud, exactitud balanceada, AUC-ROC, Average Precision, Brier Score y Log Loss. También registramos el propio umbral para referencia.



In [ ]:

metrics = {
    'accuracy': float((y_pred == y_test).mean()),
    'roc_auc': float(roc_auc_score(y_test, proba_test)),
    'avg_precision': float(average_precision_score(y_test, proba_test)),
    'brier': float(brier_score_loss(y_test, proba_test)),
    'log_loss': float(log_loss(y_test, proba_test)),
    'best_threshold': float(best_thresh)
}

pos_mask = y_test == 1
neg_mask = y_test == 0
sens = (y_pred[pos_mask] == 1).mean() if pos_mask.any() else np.nan
spec = (y_pred[neg_mask] == 0).mean() if neg_mask.any() else np.nan
metrics['balanced_acc'] = float(np.nanmean([sens, spec]))

metrics_df = pd.Series(metrics, name='XGBoost (Youden)').to_frame().T
metrics_df




## Importancia de características

Inspeccionamos rápidamente las diferencias más relevantes según la importancia de atributos calculada por XGBoost para orientar futuros análisis.



In [ ]:

feature_importance = pd.Series(xgb.feature_importances_, index=X_all.columns).sort_values(ascending=False)
feature_importance.head(25)




## Gráficas principales

Visualizamos tres perspectivas complementarias del rendimiento empleando el modelo XGBoost con el umbral de Youden:

1. **Curva ROC** para analizar el equilibrio entre verdaderos positivos y falsos positivos.
2. **Curva de calibración** para comprobar la calidad probabilística de las predicciones.
3. **Matriz de confusión** para resumir aciertos y errores con el umbral óptimo calculado.



In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

fpr_test, tpr_test, _ = roc_curve(y_test, proba_test)
axes[0].plot(fpr_test, tpr_test, label='XGBoost')
axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray', label='Azar')
axes[0].set_title('Curva ROC - XGBoost')
axes[0].set_xlabel('Falsos positivos (FPR)')
axes[0].set_ylabel('Verdaderos positivos (TPR)')
axes[0].legend()

prob_true, prob_pred = calibration_curve(y_test, proba_test, n_bins=10, strategy='quantile')
axes[1].plot(prob_pred, prob_true, marker='o', label='Calibración')
axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray', label='Ideal')
axes[1].set_xlabel('Probabilidad predicha')
axes[1].set_ylabel('Proporción real de victorias')
axes[1].set_title('Curva de calibración')
axes[1].legend()

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[2])
axes[2].set_title(f'Matriz de confusión (umbral {best_thresh:.2f})')
axes[2].set_xlabel('Predicción')
axes[2].set_ylabel('Valor real')
axes[2].set_xticklabels(['Derrota', 'Victoria'])
axes[2].set_yticklabels(['Derrota', 'Victoria'], rotation=0)

plt.tight_layout()
plt.show()




## Guardado de predicciones

Exportamos un CSV con las probabilidades y predicciones binarias del modelo XGBoost aplicando el umbral de Youden. El archivo se guarda en la carpeta indicada por `OUTPUT_DIR`, que se crea automáticamente si no existe, e incluye identificadores básicos del partido.



In [ ]:

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

preds = meta_test.reset_index(drop=True).copy()
if 'HOME_GAME_DATE' in preds.columns:
    preds = preds.rename(columns={'HOME_GAME_DATE': 'GAME_DATE'})

preds['y_true'] = y_test
preds['proba_home_win'] = proba_test
preds['pred_home_win'] = y_pred
preds['threshold_used'] = best_thresh

output_path = Path(OUTPUT_DIR) / 'test_preds.csv'
preds.to_csv(output_path, index=False)
print(f"Predicciones guardadas en {output_path}")




## Conclusiones y próximos pasos

1. Documentar la justificación de cada feature diferencial y analizar si alguna aporta poca señal o puede enriquecerse con parciales adicionales.
2. Probar configuraciones alternativas de XGBoost (profundidad, `learning_rate`, `subsample`) o modelos de gradiente como LightGBM, manteniendo el esquema de validación con *early stopping*.
3. Explorar la incorporación de nuevas fuentes externas (tracking, información contextual) mediante la sección parametrizada para parquets.
4. Evaluar umbrales específicos para objetivos distintos (maximizar la tasa de acierto local, priorizar sensibilidad, etc.) a partir de las curvas ROC/PR.
5. Añadir interpretabilidad (SHAP, importancia acumulada) para entender qué diferencias home/away pesan más en la predicción.
6. Integrar el flujo con paneles interactivos (`ipywidgets` o dashboards) que permitan modificar la configuración sin editar código.

